In [1]:
import pandas as pd
import numpy as np

from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate
from surprise import accuracy
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
df_train = pd.read_csv('../000.data/train/train_D.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_D" のテストデータだけを抽出
df_test = test[test["user_id"].str.endswith("_D")]

In [4]:
# 広告経由の購入のみ関連度3、それ以外はスコアをそのまま使用
def calculate_relevance(row):
    if row['event_type'] == 3 and row['ad'] == 1:
        return 3
    return row['event_type']

df_train['rating'] = df_train.apply(calculate_relevance, axis=1)

In [5]:
# Surprise用データセット形式に変換
reader = Reader(rating_scale=(0, 3))
data = Dataset.load_from_df(df_train[['user_id', 'product_id', 'rating']], reader)

In [6]:
# クロスバリデーション実行
model = SVD()
cv_results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.3917  0.3911  0.3918  0.3920  0.3923  0.3918  0.0004  
MAE (testset)     0.2985  0.2981  0.2984  0.2985  0.2987  0.2984  0.0002  
Fit time          31.78   31.63   33.11   30.59   27.69   30.96   1.82    
Test time         5.12    4.01    5.10    5.04    3.59    4.57    0.65    


In [7]:
# nDCG計算
def calculate_ndcg(predictions, k=22):
    def dcg(scores):
        return sum([score / np.log2(idx + 2) for idx, score in enumerate(scores)])

    user_ranking = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_ranking[uid].append((iid, est))

    ndcg_values = []
    for uid, user_preds in user_ranking.items():
        user_preds.sort(key=lambda x: x[1], reverse=True)
        relevance_scores = [3 if i < k else 0 for i, _ in enumerate(user_preds)]
        ideal_scores = sorted(relevance_scores, reverse=True)
        ndcg = dcg(relevance_scores) / dcg(ideal_scores) if dcg(ideal_scores) > 0 else 0
        ndcg_values.append(ndcg)

    return np.mean(ndcg_values)

In [8]:
# nDCGの評価を実施
full_trainset = data.build_full_trainset()
model.fit(full_trainset)
testset = full_trainset.build_testset()
predictions = model.test(testset)
print(f"nDCG Score: {calculate_ndcg(predictions):.4f}")

nDCG Score: 1.0000


In [9]:
# 提出用データ作成
def create_submission(model, user_ids, k=22):
    result = []
    for uid in user_ids:
        all_predictions = [model.predict(uid, iid) for iid in df_train['product_id'].unique()]
        all_predictions.sort(key=lambda x: x.est, reverse=True)
        top_k = all_predictions[:k]
        result.extend([[uid, pred.iid, rank + 1] for rank, pred in enumerate(top_k)])
    return pd.DataFrame(result, columns=['user_id', 'product_id', 'rank'])

In [10]:
user_ids = df_test['user_id'].unique()
submission_df = create_submission(model, user_ids)

In [ ]:
submission_df.to_csv('./predict_file/recommendation_result_D.tsv', sep="\t", index=False)

: 